In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import h5py
import pandas as pd

## Add the path to the DiscEvolution directory
sys.path.append('/home/bentobin/code/DiscEvolution')

## import DiscEvolution modules
from DiscEvolution.chemistry import SimpleCOMolAbund

## Population Synthesis Directory

In [ ]:
MODEL = 'testmodel'
OUTPUT_DIR = f'/home/bentobin/simulations/{MODEL}/output/'
PKL_FILE = f'/home/bentobin/simulations/{MODEL}/df_planets_{MODEL}.pkl'
FIG_DIR = f'/home/bentobin/plots/{MODEL}/'
SAVE_FIGS = True

## Global Plotting Style

In [ ]:
## Plotting imports
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.lines import Line2D

## Global plot style

plt.rcParams.update({
    'axes.linewidth': 2.0,
    'axes.labelsize': 14,
    'axes.labelweight': 'normal',
    'xtick.major.width': 2.0,
    'ytick.major.width': 2.0,
    'xtick.minor.width': 1.5,
    'ytick.minor.width': 1.5,
    'xtick.major.size': 6,
    'ytick.major.size': 6,
    'xtick.minor.size': 4,
    'ytick.minor.size': 4,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12})

## Method to bold the tick labels

def bold_ticks(*axes):
    for ax in axes:
        labels = (ax.get_xticklabels() + ax.get_yticklabels()
            + ax.get_xticklabels(minor = True) + ax.get_yticklabels(minor = True))

        for label in labels:
            label.set_fontweight('bold')

## Method to Load in Data

In [ ]:
## Method to read data from an HDF5 dataset or group

def read_hdf5_array(obj):
    """
    Read data from either an HDF5 dataset or an HDF5 group.
    """

    ## Normal case (the data are stored as one HDF5 dataset)
    if isinstance(obj, h5py.Dataset):
        return np.array(obj)

    ## Ragged case (the data are stored as a group)
    elif isinstance(obj, h5py.Group):
        values = []

        for key in sorted(obj.keys(), key = int):
            values.append(np.array(obj[key]))

        return values

    ## Raise an error if an unexpected HDF5 object is encountered
    else:
        raise TypeError(f"Unexpected HDF5 object type: {type(obj)}")

## Method to load data from an HDF5 file

def load_data(data_dir):
    """
    Load a disc evolution output file (HDF5 only).

    Parameters:
        data_dir: Path to the HDF5 output file.

    Returns:
        data: Dictionary containing the simulation output.
    """

    data = {}

    ## Open the HDF5 file in read-only mode
    with h5py.File(data_dir, "r") as f:

        ## -----------------------
        ## General simulation data
        ## -----------------------

        general_keys = [
            "R",
            "t",
            "disk_Mdot_star",
            "disk_Mass",
            "Tc",
            "Sigc"]

        for key in general_keys:
            ## Only load the quantity if it exists in the file
            if key in f:
                data[key] = np.array(f[key])

        ## -----------------
        ## Scalar attributes
        ## -----------------

        scalar_attributes = [
            "alpha_SS",
            "psi_DW",
            "Mdot",
            "Mdisk",
            "Rd",
            "pla_eff",
            "f_plt",
            "complete",
            "aborted"]

        for key in scalar_attributes:
            ## Only load the attribute if it exists in the file
            if key in f.attrs:
                data[key] = f.attrs[key].item()

        ## Names of the chemical species (only written when chemistry is on)
        if "chem_species" in f.attrs:
            data["chem_species"] = [s.decode() if isinstance(s, bytes) else str(s) for s in f.attrs["chem_species"]]

        ## -------------
        ## Disk profiles
        ## -------------

        disk_keys = [
            "T",
            "time_snap",
            "Sigma_G",
            "Sigma_dust",
            "Sigma_pebbles",
            "Sigma_planetesimals",
            "Vdrift_grains",
            "Vdrift_pebbles",
            "St_grains",
            "St_pebbles",
            "St_planetesimals",
            "e_planetesimals",
            "i_planetesimals",
            "de2_dt",
            "di2_dt",
            "de2_dt_drag",
            "di2_dt_drag",
            "de2_dt_VS_embryo",
            "di2_dt_VS_embryo",
            "de2_dt_VS_pltsml",
            "di2_dt_VS_pltsml",
            "de2_dt_DF",
            "di2_dt_DF",
            "gas_chem",
            "ice_chem",
            "planetesimal_ice_chem"]

        for key in disk_keys:
            ## Only load the quantity if it exists in the file
            if key in f:
                data[key] = read_hdf5_array(f[key])

        ## -----------
        ## Planet data
        ## -----------

        planet_keys = [
            "Mcs",
            "Mes",
            "Rp",
            "disk_Mdot_p",
            "Mdot_planetesimal",
            "Mdot_pebble_core",
            "Mdot_pebble_env",
            "Mdot_migration",
            "Mdot_gas",
            "M_iso_planetesimal",
            "M_iso_pebble",
            "X_cores",
            "X_envs",
            "f_env",
            "t_form"]

        for key in planet_keys:
            ## Only load the quantity if it exists in the file
            if key in f:
                data[key] = read_hdf5_array(f[key])

    return data

## Build Planet Dataframe

In [ ]:
def build_planet_dataframe(output_dir, pkl_file):
    """
    Loop over every completed HDF5 output file, store the final state of each
    planet (plus its initial position, implant time and the simulation metadata)
    as one row of a dataframe, and save the dataframe as a pickle file.

    Parameters:
        output_dir: Directory containing the .h5 output files.
        pkl_file: Path to save the pickled dataframe to.

    Returns:
        df_planets: Dataframe with one row per planet.
    """

    planet_rows = []
    chem_species = None
    n_files = 0

    ## ----------------------
    ## Main loop over files
    ## ----------------------

    for fname in sorted(os.listdir(output_dir)):

        if not fname.endswith(".h5"):
            continue

        fpath = os.path.join(output_dir, fname)

        try:
            ## Load raw simulation output
            data = load_data(fpath)

            ## Only use simulations that have completed
            if not bool(data.get("complete", False)):
                continue

            ## Skip if no planets
            if "Mcs" not in data or len(data["Mcs"]) == 0:
                continue

            n_files += 1
            chem_species = data.get("chem_species", chem_species)

            ## Simulation metadata from the HDF5 attributes (shared by every planet in this file)
            metadata = {
                "Mdisk": data.get("Mdisk", np.nan),
                "Mdot": data.get("Mdot", np.nan),
                "Rd": data.get("Rd", np.nan),
                "pla_eff": data.get("pla_eff", np.nan),
                "f_plt": data.get("f_plt", np.nan),
                "alpha_SS": data.get("alpha_SS", np.nan),
                "psi_DW": data.get("psi_DW", np.nan),
                "last_t": float(data["t"][-1]),
            }

            Mcs = np.array(data["Mcs"])
            Mes = np.array(data["Mes"])
            Rps = np.array(data["Rp"])
            t_form = np.array(data["t_form"])
            X_cores = np.array(data["X_cores"]) if "X_cores" in data else None
            X_envs = np.array(data["X_envs"]) if "X_envs" in data else None

            ## Loop planets
            for ip in range(len(Mcs)):
                planet_rows.append({
                    ## Identifiers
                    "file": fname,
                    "planet_index": ip,

                    ## Final masses and position
                    "Mc_final": Mcs[ip, -1],
                    "Me_final": Mes[ip, -1],
                    "Mp_final": Mcs[ip, -1] + Mes[ip, -1],
                    "Rp_final": Rps[ip, -1],

                    ## Final chemistry
                    "X_core_final": X_cores[ip, :, -1] if X_cores is not None else None,
                    "X_env_final": X_envs[ip, :, -1] if X_envs is not None else None,

                    ## Implantation
                    "Rp_init": Rps[ip][~np.isnan(Rps[ip])][0],
                    "t_implant": float(t_form[ip]),

                    ## Simulation metadata
                    **metadata,
                })

        except Exception as e:
            print(f"Error in file {fname}: {e}")

    ## Convert to dataframe and save to pkl file
    df_planets = pd.DataFrame(planet_rows)
    df_planets.attrs["chem_species"] = chem_species

    print("Processed", n_files, "completed files")
    print("Planets:", df_planets.shape)

    df_planets.to_pickle(pkl_file)

    return df_planets

df_planets = build_planet_dataframe(OUTPUT_DIR, PKL_FILE)

## Build Disk Parameter Dataframe

In [ ]:
def load_disc_attributes(output_dir, completed_only=True):
    """
    Read the disc parameters stored as attributes in every HDF5 output file.

    Parameters:
        output_dir: Directory containing the .h5 output files.
        completed_only: If True, only include simulations that have completed.

    Returns:
        df_discs: Dataframe with one row per simulated disc.
    """

    disc_rows = []

    grid_keys = ["Mdisk", "Mdot", "Rd", "alpha_SS", "psi_DW", "pla_eff", "f_plt"]

    for fname in sorted(os.listdir(output_dir)):

        if not fname.endswith(".h5"):
            continue

        with h5py.File(os.path.join(output_dir, fname), "r") as f:

            if completed_only and not bool(f.attrs.get("complete", False)):
                continue

            disc_rows.append({
                "file": fname,
                **{key: f.attrs[key].item() for key in grid_keys if key in f.attrs},
            })

    return pd.DataFrame(disc_rows)

df_discs = load_disc_attributes(OUTPUT_DIR)

## Plot Disc Parameter Space

In [ ]:
## Plot solved psi_DW against initial Mdot for each simulated disc
## sized by Rd, coloured by Mdisk, marker style by alpha_SS

fig, ax = plt.subplots(figsize=(8, 6))

## Colour: one colour per Mdisk value in the grid
## The saved Mdisk is measured from the disc, so round to 3 decimal places to group the grid values
Mdisk_rounded = df_discs["Mdisk"].round(3)
Mdisk_values = np.sort(Mdisk_rounded.unique())

if len(Mdisk_values) > len(plt.cm.tab10.colors):
    raise ValueError(f"{len(Mdisk_values)} Mdisk values but tab10 only has {len(plt.cm.tab10.colors)} colours")

## Each disc is coloured by the index of its Mdisk value, with one colour bin per index
Mdisk_index = pd.Series(np.searchsorted(Mdisk_values, Mdisk_rounded), index = df_discs.index)
cmap = ListedColormap(plt.cm.tab10.colors[:len(Mdisk_values)])
norm = BoundaryNorm(np.arange(len(Mdisk_values) + 1) - 0.5, cmap.N)

## Size: marker area scales linearly with Rd
Rd_min, Rd_max = df_discs["Rd"].min(), df_discs["Rd"].max()
size_range = (30, 250)

def Rd_to_size(Rd):
    if Rd_max == Rd_min:
        return np.full_like(np.asarray(Rd, dtype = float), np.mean(size_range))
    
    return size_range[0] + (np.asarray(Rd) - Rd_min) / (Rd_max - Rd_min) * (size_range[1] - size_range[0])

## Marker: one style per alpha_SS value
markers = ["o", "s", "^", "D", "v", "P", "X"]
alphas = np.sort(df_discs["alpha_SS"].unique())

## Initial accretion rate in Msun/Myr (the saved Mdot is in Msun/yr)
Mdot_Myr = df_discs["Mdot"] * 1e6

## The saved Mdot is measured from the disc, so round to 2 significant figures to group the grid values
Mdot_rounded = [float(f"{Mdot:.2g}") for Mdot in Mdot_Myr]
Mdot_values = np.unique(Mdot_rounded)

## x-axis: each disc is placed at the index of its Mdot value (0, 1, 2, ...) so the Mdot values are evenly spaced
Mdot_index = pd.Series(np.searchsorted(Mdot_values, Mdot_rounded), index = df_discs.index)

## Small random sideways offset within each Mdot column so overlapping points can be seen
rng = np.random.default_rng(42)
x_plot = Mdot_index + rng.uniform(-0.28, 0.28, size = len(Mdot_index))

for i, alpha in enumerate(alphas):
    df_a = df_discs[df_discs["alpha_SS"] == alpha]
    ax.scatter(x_plot[df_a.index], df_a["psi_DW"],
               s = Rd_to_size(df_a["Rd"]), c = Mdisk_index[df_a.index], cmap = cmap, norm = norm,
               marker = markers[i % len(markers)], edgecolors = "white", linewidths = 0.8, zorder = 3)

## One tick per Mdot value, with half a step of space at each end
ax.set_xticks(np.arange(len(Mdot_values)), [f"{Mdot:g}" for Mdot in Mdot_values])
ax.set_xlim(-0.5, len(Mdot_values) - 0.5)

if (df_discs["psi_DW"] > 0).all():
    ax.set_yscale("log")

ax.set_xlabel(r"Initial $\dot{M}$ [M$_\odot$ Myr$^{-1}$]")
ax.set_ylabel(r"Solved $\psi_{\rm DW}$")
ax.grid(True, which = "major", color = "0.9", linewidth = 0.8, zorder = 0)

## Discrete colourbar for Mdisk, with one labelled block per value
cbar = fig.colorbar(plt.cm.ScalarMappable(norm = norm, cmap = cmap), ax = ax, ticks = np.arange(len(Mdisk_values)))
cbar.set_ticklabels([f"{M:g}" for M in Mdisk_values])
cbar.ax.minorticks_off()
cbar.set_label(r"$M_{\rm disk}$ [M$_\odot$]")

## Single legend: alpha_SS (marker style) followed by Rd (marker size)
alpha_handles = [Line2D([], [], marker = markers[i % len(markers)], linestyle = "", color = "0.4", markersize = 8,
                 label = rf"$\alpha_{{\rm SS}}$ = {alpha:.0e}") for i, alpha in enumerate(alphas)]

Rd_values = np.sort(df_discs["Rd"].round(0).unique())

if len(Rd_values) > 4:
    Rd_values = np.unique(np.round(np.linspace(Rd_min, Rd_max, 4), 0))

## Line2D markersize is a diameter in points, scatter s is an area in points^2, so use the square root
Rd_handles = [Line2D([], [], marker = "o", linestyle = "", color = "0.6", markeredgecolor = "white",
              markersize = np.sqrt(Rd_to_size(Rd)), label = rf"$R_{{\rm d}}$ = {Rd:.0f} au") for Rd in Rd_values]

ax.legend(handles = alpha_handles + Rd_handles, loc = "upper left", labelspacing = 1.0)

fig.tight_layout()

## Bold the tick labels on the plot and the colourbar
bold_ticks(ax, cbar.ax)

## Save the figure to FIG_DIR
if SAVE_FIGS:
    os.makedirs(FIG_DIR, exist_ok = True)
    fig.savefig(os.path.join(FIG_DIR, "disc_parameter_space.png"), dpi = 300, bbox_inches = "tight")

plt.show()

## Bulk Composition Functions

In [ ]:
## Method to load the final core and envelope abundances of every planet

def final_abundances(df_planets):
    """
    Load the final core and envelope abundances of every planet in the dataframe.

    Parameters:
        df_planets: Planet dataframe from build_planet_dataframe().

    Returns:
        core: SimpleCOMolAbund with one column per planet (final core abundances).
        env: SimpleCOMolAbund with one column per planet (final envelope abundances).
        Mc: Final core masses with shape (N_planets,).
        Me: Final envelope masses with shape (N_planets,).
    """

    n_planets = len(df_planets)

    core = SimpleCOMolAbund(n_planets)
    env = SimpleCOMolAbund(n_planets)

    ## The species order in the HDF5 files must match SimpleCOMolAbund
    chem_species = df_planets.attrs.get("chem_species")
    
    if chem_species is not None and list(chem_species) != list(core.names):
        raise ValueError(f"Species order {chem_species} does not match SimpleCOMolAbund {list(core.names)}")

    ## Stack the per-planet arrays into shape (N_species, N_planets)
    core.data[:] = np.stack(df_planets["X_core_final"].to_numpy(), axis = 1)
    env.data[:] = np.stack(df_planets["X_env_final"].to_numpy(), axis = 1)

    Mc = df_planets["Mc_final"].to_numpy(dtype = float)
    Me = df_planets["Me_final"].to_numpy(dtype = float)

    return core, env, Mc, Me

## Method to calculate the final bulk metallicity of each planet

def bulk_metallicity(df_planets):
    """
    Calculate the final bulk metallicity of each planet.
    This is the mass fraction of the whole planet (core + envelope) in heavy elements.

    Parameters:
        df_planets: Planet dataframe from build_planet_dataframe().

    Returns:
        Z: Bulk metallicity, as a Series aligned with df_planets.
    """

    core, env, Mc, Me = final_abundances(df_planets)

    ## Mass of heavy elements in the core and envelope
    M_Z = core.total_abund * Mc + env.total_abund * Me

    Z = M_Z / (Mc + Me)

    return pd.Series(Z, index = df_planets.index)

## Method to calculate the final envelope metallicity of each planet

def envelope_metallicity(df_planets):
    """
    Calculate the final envelope metallicity of each planet.
    This is the mass fraction of the envelope in heavy elements (the rest is H/He).

    Parameters:
        df_planets: Planet dataframe from build_planet_dataframe().

    Returns:
        Z_env: Envelope metallicity, as a Series aligned with df_planets.
    """

    core, env, Mc, Me = final_abundances(df_planets)

    Z_env = env.total_abund

    return pd.Series(Z_env, index = df_planets.index)

## Method to calculate the final volatile to refractory ratio of each planet

def bulk_volatile_refractory_ratio(df_planets):
    """
    Calculate the final bulk volatile to refractory mass ratio of each planet.
    Volatiles are CO, CH4, CO2 and H2O and refractories are C-grain and Si-grain.

    Parameters:
        df_planets: Planet dataframe from build_planet_dataframe().

    Returns:
        VR: Volatile to refractory ratio, as a Series aligned with df_planets.
            NaN wherever there are no refractories.
    """

    core, env, Mc, Me = final_abundances(df_planets)

    ## Mass of volatiles and refractories in the core and envelope
    volatiles = ((core["CO"] + core["CH4"] + core["CO2"] + core["H2O"]) * Mc
        + (env["CO"] + env["CH4"] + env["CO2"] + env["H2O"]) * Me)

    refractories = ((core["C-grain"] + core["Si-grain"]) * Mc
        + (env["C-grain"] + env["Si-grain"]) * Me)

    with np.errstate(invalid = "ignore", divide = "ignore"):
        VR = volatiles / refractories

    VR = np.where(np.isfinite(VR), VR, np.nan)

    return pd.Series(VR, index = df_planets.index)

## Method to calculate the final bulk water content of each planet

def bulk_water_content(df_planets):
    """
    Calculate the final bulk water content of each planet.
    This is the mass fraction of the whole planet (core + envelope) that is H2O.

    Parameters:
        df_planets: Planet dataframe from build_planet_dataframe().

    Returns:
        f_H2O: Water mass fraction, as a Series aligned with df_planets.
    """

    core, env, Mc, Me = final_abundances(df_planets)

    ## Mass of water in the core and envelope
    M_H2O = core["H2O"] * Mc + env["H2O"] * Me

    f_H2O = M_H2O / (Mc + Me)

    return pd.Series(f_H2O, index = df_planets.index)

## Plot Final Bulk Compositions

In [ ]:
## Plot the final bulk composition of each planet against its total mass,
## one figure per quantity, coloured by final position

## Final bulk composition of each planet
composition = [
    (bulk_metallicity(df_planets), r"Bulk Metallicity", "linear", "bulk_metallicity.png"),
    (envelope_metallicity(df_planets), r"Envelope Metallicity", "linear", "envelope_metallicity.png"),
    (bulk_volatile_refractory_ratio(df_planets), "Volatile / Refractory Ratio", "log", "volatile_refractory_ratio.png"),
    (bulk_water_content(df_planets), r"Bulk Water Content", "log", "bulk_water_content.png")]

Mp = df_planets["Mp_final"]
Rp = df_planets["Rp_final"]

## Colour: final position on a linear scale with the RdYlBu colormap (red = close-in, blue = far out)
## The same colour scale is used for all four figures so they can be compared directly
cmap = plt.cm.RdYlBu
norm = plt.Normalize(vmin = Rp.min(), vmax = Rp.max())

for values, ylabel, yscale, fig_name in composition:

    fig, ax = plt.subplots(figsize = (8, 6))

    ax.scatter(Mp, values, c = Rp, cmap = cmap, norm = norm, s = 30,
               edgecolors = "white", linewidths = 0.5, zorder = 3)

    ax.set_xscale("log")
    ax.set_yscale(yscale)
    ax.set_xlabel(r"Total Planet Mass [M$_\oplus$]")
    ax.set_ylabel(ylabel)
    ax.grid(True, which = "major", color = "0.9", linewidth = 0.8, zorder = 0)

    ## Colourbar for final position
    cbar = fig.colorbar(plt.cm.ScalarMappable(norm = norm, cmap = cmap), ax = ax)
    cbar.set_label(r"Final Position [AU]")

    fig.tight_layout()

    ## Bold the tick labels on the plot and the colourbar
    bold_ticks(ax, cbar.ax)

    ## Save the figure to FIG_DIR
    if SAVE_FIGS:
        os.makedirs(FIG_DIR, exist_ok = True)
        fig.savefig(os.path.join(FIG_DIR, fig_name), dpi = 300, bbox_inches = "tight")

    plt.show()